### What this measures

Two questions that `ganapati` needs answers to, and neither can be settled by reading
the code:

1. **Which encoder should the Sanskrit shelf use?** Devanagari is where a static lookup table is
   weakest — sandhi and compounding mean the words a reader types are rarely the tokens on the
   page — so this is exactly the case where paying for a transformer might be justified.
2. **What does the lexicon layer buy?** `vidyut_pipe` writes lemmas and `mw_lexicon` writes English
   glosses into `metadata`, which costs an 81 MB download and some ingest time. Is that worth more
   or less than changing the encoder?

**The corpus is the whole Bhagavad Gītā** — 701 verses, indexed as Devanagari and nothing else.
Queries come from two independent published English translations, so an English query is a genuine
cross-lingual retrieval with no possibility of leakage: there is no English in the index. Swami
Sivananda stays close to the text; Shri Purohit Swami is a free paraphrase. The pair brackets easy
and hard.

Everything is scored as MRR and top-1 through the **hybrid** FTS + vector RRF pipeline that
actually runs, not by cosine similarity alone. That distinction matters here more than usual: the
script-folding FTS5 tokenizer already answers the Devanagari and IAST legs for free, so a pure
vector score credits the encoder for work it does not do.

In [ ]:
#| hide
from nbdev.showdoc import *

## The corpus

701 verses with five independent English translations each, from the
[gita/gita](https://github.com/gita/gita) dataset. The verse text and its IAST transliteration go
in; the translations are held out entirely and used only as queries.

In [ ]:
#| eval: false
import json, random, time, urllib.request
from pathlib import Path
import numpy as np

SRC = 'https://raw.githubusercontent.com/gita/gita/main/data'
def fetch(name):
    'Cache one dataset file beside the notebook.'
    p = Path(f'bench/{name}'); p.parent.mkdir(exist_ok=True)
    if not p.exists():
        req = urllib.request.Request(f'{SRC}/{name}', headers={'User-Agent':'litesearch'})
        with urllib.request.urlopen(req, timeout=120) as r: p.write_bytes(r.read())
    return json.loads(p.read_text())

V, T = fetch('verse.json'), fetch('translation.json')
ids  = [v['id'] for v in V if (v.get('text') or '').strip()]
DEVA = {v['id']: v['text'].replace('\n',' ').strip() for v in V}
IAST = {v['id']: (v.get('transliteration') or '').replace('\n',' ').strip() for v in V}
EN   = {}
for t in T:
    if t.get('lang') == 'english': EN.setdefault(t['authorName'], {})[t['verse_id']] = (t.get('description') or '').strip()
print(len(ids), 'verses |', {a: len(v) for a, v in EN.items()})

## Four query types

The first two are lexical and the FTS leg should carry them; the last two are the ones that decide
anything, because nothing in the index shares a character with them.

In [ ]:
#| eval: false
NQ = 250
A_CLOSE, A_FREE = 'Swami Sivananda', 'Shri Purohit Swami'
rng = random.Random(0)
QSETS = {'Deva→verse'    : [(i, DEVA[i]) for i in rng.sample(ids, NQ)],
         'IAST→verse'    : [(i, IAST[i]) for i in ids if IAST.get(i)][:NQ],
         'EN close→verse': [(i, EN[A_CLOSE][i]) for i in ids if EN[A_CLOSE].get(i)][:NQ],
         'EN free→verse' : [(i, EN[A_FREE][i])  for i in ids if EN[A_FREE].get(i)][:NQ]}
{k: len(v) for k, v in QSETS.items()}

## Scoring

One store per encoder, every verse a document, hybrid search with the encoder's own query vector.

In [ ]:
#| eval: false
from litesearch import database, static_embedder, doc_encoder, query_encoder

def mk(spec):
    'A (doc, query, dims) triple for a model2vec id/path or a litesearch ONNX model dict.'
    m = static_embedder() if isinstance(spec, dict) else static_embedder(spec)
    f16 = lambda fn: lambda xs: np.asarray(fn(xs), dtype=np.float16)
    return f16(doc_encoder(m)), f16(query_encoder(m)), int(np.asarray(m.encode(['x'])).shape[-1])

def build(spec):
    d, q, dims = mk(spec)
    t0 = time.time(); db = database(); st = db.get_store(ndim=dims, dtype=np.float16)
    st.insert_all([{'content': DEVA[i], 'metadata': json.dumps({'id': i})} for i in ids])
    st.update_embeddings(d)
    return db, q, dims, time.time()-t0

def score(db, q, qs, k=10):
    'MRR@k and top-1 for one query set.'
    t1 = rr = 0.0
    for vid, text in qs:
        hits = db.search(text, q([text])[0].tobytes(), columns=['metadata'], limit=k, dtype=np.float16)
        rank = [j+1 for j, h in enumerate(hits) if json.loads(h['metadata'])['id'] == vid]
        b = min(rank) if rank else k+1
        t1 += b == 1; rr += 1/b
    return rr/len(qs), t1/len(qs)

## Encoders

Two static models at the same width and one ONNX transformer for the ceiling. `potion-multilingual`
is litesearch's default; `distilled-embeddinggemma` is a Model2Vec distillation of
`google/embeddinggemma-300m`, chosen because its teacher is multilingual and its tokenizer carries
13,754 Devanagari entries.

In [ ]:
#| eval: false
ENCS = [('potion-multilingual', 'minishlab/potion-multilingual-128M'),
        ('distilled-256',       'karthikrajgopal/distilled-embeddinggemma'),
        ('embgemma-512',        'staticgemma-512'),      # same teacher, twice the width
        ('sanskritgemma-256',   'sanskritgemma-256'),    # Sanskrit-tuned teacher, Matryoshka widths
        ('sanskritgemma-768',   'sanskritgemma-768'),
        ('embedding_gemma',     embedding_gemma)]   # needs litesearch's evals/onnx.py
rows = {}
for nm, spec in ENCS:
    db, q, dims, ti = build(spec)
    rows[nm] = {k: score(db, q, qs) for k, qs in QSETS.items()}
    print(f'{nm:22s} {dims:4d}d  ingest {ti:6.1f}s  ' +
          '  '.join(f'{k} {rows[nm][k][0]:.3f}' for k in QSETS), flush=True)

## Does the lexicon layer beat changing the encoder?

`vidyut_pipe` writes lemmas and `mw_lexicon` writes English glosses into `metadata`, which
`get_store` already indexes for FTS beside `content`. Neither is embedded, so the whole gain — if
there is one — lands on the keyword leg.

In [ ]:
#| eval: false
from ganapati import vidyut_pipe, mw_lexicon, lemma_facets, gloss_facets
nlp, mw = vidyut_pipe(), mw_lexicon()

def build_faceted(spec, lemma=True, gloss=True):
    d, q, dims = mk(spec)
    db = database(); st = db.get_store(ndim=dims, dtype=np.float16)
    rows_ = []
    for i in ids:
        md = {'id': i}
        if lemma: md |= lemma_facets(DEVA[i], nlp)
        if gloss: md |= gloss_facets(DEVA[i], nlp, mw)
        rows_.append({'content': DEVA[i], 'metadata': json.dumps(md, ensure_ascii=False)})
    st.insert_all(rows_); st.update_embeddings(d)
    return db, q

base = 'karthikrajgopal/distilled-embeddinggemma'
for label, kw in (('lemma only', dict(gloss=False)), ('gloss only', dict(lemma=False)), ('both', {})):
    db, q = build_faceted(base, **kw)
    print(f'{label:12s} ' + '  '.join(f'{k} {score(db, q, qs)[0]:.3f}' for k, qs in QSETS.items()), flush=True)

## What the numbers said

All runs over the whole Gītā. Encoder-only rows index each verse as its own document; full-stack
rows ingest 18 GRETIL-style chapter files through `add_file`, so `VerseChunker`, verse tree mode
and the metre facets all apply.

### 1. Encoder only — 701 documents, relevance is the verse

| encoder | teacher | dims | Deva | IAST | EN close | EN free | mean |
|---|---|---|---|---|---|---|---|
| potion-multilingual | — | 256 | 1.000 | 0.647 | 0.265 | 0.233 | 0.536 |
| embgemma-256 | embeddinggemma-300m | 256 | 1.000 | 0.684 | 0.251 | 0.205 | 0.535 |
| embgemma-512 | embeddinggemma-300m | 512 | 1.000 | 0.677 | 0.243 | 0.203 | 0.531 |
| sanskritgemma-256 | rgveda-embedding-gemma | 256 | 1.000 | 0.656 | 0.317 | 0.249 | 0.555 |
| sanskritgemma-384 | rgveda-embedding-gemma | 384 | 1.000 | 0.666 | 0.325 | 0.255 | 0.561 |
| sanskritgemma-512 | rgveda-embedding-gemma | 512 | 1.000 | 0.669 | 0.326 | 0.258 | 0.563 |
| sanskritgemma-768 | rgveda-embedding-gemma | 768 | 1.000 | 0.672 | 0.334 | 0.255 | **0.565** |
| embedding_gemma (ONNX) | — | 768 | 1.000 | 0.904 | 0.416 | 0.318 | **0.659** |

### 2. Full stack — `find` / `sections` / `context` / `context + graph`, relevance is the verse

| encoder | close:find | close:sect | close:ctx | free:find | free:sect | free:ctx | mean |
|---|---|---|---|---|---|---|---|
| potion-multilingual | 0.190 | 0.210 | **0.340** | 0.103 | 0.129 | **0.296** | **0.238** |
| sanskritgemma-768 | 0.183 | 0.222 | 0.337 | 0.076 | 0.113 | 0.231 | 0.216 |
| sanskritgemma-256 | 0.184 | 0.211 | 0.326 | 0.071 | 0.123 | 0.233 | 0.213 |
| embgemma-512 | 0.150 | 0.201 | 0.342 | 0.057 | 0.090 | 0.217 | 0.202 |
| embgemma-256 | 0.155 | 0.176 | 0.341 | 0.064 | 0.080 | 0.220 | 0.200 |

### Six things worth taking away

**The lexical legs belong to the tokenizer, not the encoder.** Every encoder scores 1.000 on
Devanagari→verse, because the script-folding FTS5 tokenizer already collapses `श्रीमाता`,
`śrīmātā` and `srimata` onto one key. Choosing an encoder on that column measures nothing.

**Corpus size changes the answer.** On a 24-verse sample `embgemma-256` looked clearly ahead of the
default (0.757 against 0.735). At 701 verses they are 0.535 and 0.536 — a dead heat. A small
retrieval benchmark flatters whichever model happens to suit it.

**Width is not the lever; the teacher is.** `embgemma-512` doubles `embgemma-256`'s dimensions from
the same teacher and scores *worse*. Re-distilling from a Sanskrit-tuned teacher at the same width
moves English-close from 0.251 to 0.326. And the Sanskrit family is nearly flat across widths —
256 reaches 98% of 768 at a third the disk, which is the Matryoshka structure showing through.
Ship the 256.

**The stack matters more than the encoder.** `context` roughly doubles `find` for every model
(0.190 → 0.340 for the best). Assembling whole sections recovers verses whose individual chunk
ranked poorly — a larger effect than any encoder swap measured here.

**The encoder-only ranking does not survive the full stack, and the full stack is the one that
counts.** `sanskritgemma` wins the encoder-only table and `potion-multilingual` wins the full-stack
table at both chapter and verse granularity. The Sanskrit tuning is real — it beats both general
`embgemma` distillations consistently — it simply does not overturn the default once the tree and
context layers are doing their work. Benchmark the configuration you ship.

**The graph leg contributes nothing here.** `context + graph` is identical to `context` to three
decimals: with verse-level relevance the operative sections already contain the answer, and
`related` adds neighbours that are not it. It earns its place on corpora where the question is
*what connects to what*, not *where is this verse*.

### A bug this eval found

`build_graph` returned **0 entities on the entire Gītā**. Not a Sanskrit problem: `_norm`, which
every entity name passes through, ended with `if not re.search(r'[A-Za-z]', s): return None`. The
intent — drop pure numbers and punctuation — is right, but spelled in ASCII it discarded every
non-Latin script, so Devanagari, Cyrillic and Han entities all normalised to `None`. The term
extractor was fine; nothing survived normalisation. Fixed to test for any Unicode letter, after
which the same corpus yields 2,899 entities and 4,093 mentions.

Worth stating plainly because of how it hid: every layer reported success, the graph was simply
empty, and no encoder benchmark would ever have touched it.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()